# Stage 1 — Is the Chatbot Getting It Right?

This notebook is the hands-on companion to **Stage 1** of the RAG chatbot evaluation plan:
measure whether the chatbot's answers match known-good reference answers, treating the
chatbot as a **black box**. No changes to the chatbot are needed — just questions, answers,
and an API call.

**How it works:**

1. Curate a test set of questions with reference answers.
2. Ask the chatbot each question and record its responses.
3. Upload the resulting dataset to S3.
4. Submit a RAGAS evaluation job to **EvalHub** — a *judge* LLM grades each answer
   against the reference.
5. Read the scores, then read the failures.

**What we measure** (all scores range 0–1, higher is better):

| Metric | Question it answers |
|--------|--------------------|
| `factual_correctness` | Do the factual claims in the answer overlap with the reference? |
| `answer_similarity` | Is the answer semantically similar to the reference? |
| `nv_accuracy` | Judged head-to-head, does the answer agree with the reference? |

Together these approximate RAGAS "answer correctness": factual overlap + semantic similarity,
which gives you more signal than a binary right/wrong.

> **Prerequisites:** the platform setup in [`README.md`](README.md) is done — EvalHub is
> deployed with the `ragas` provider, MinIO (or other S3) is available, and the
> `judge-model-auth` and `evalhub-test-data-s3` secrets exist in the tenant namespace.


In [ ]:
%pip install -q "eval-hub-sdk[client]" boto3 openai pandas requests python-dotenv


## Configuration

Point this at your cluster and your chatbot. Everything can be overridden with
environment variables (a `.env` file next to this notebook is also picked up), so no
credentials need to live in the notebook itself.


In [ ]:
import os
import subprocess

from dotenv import load_dotenv

load_dotenv()  # optionally pick up a local .env file


def _oc(*args: str) -> str:
    return subprocess.run(["oc", *args], capture_output=True, text=True).stdout.strip()


# --- EvalHub service ---
EVALHUB_URL = os.environ.get("EVALHUB_URL") or "https://" + _oc(
    "get", "route", "evalhub", "-n", "evalhub", "-o", "jsonpath={.spec.host}"
)
EVALHUB_TENANT = os.environ.get("EVALHUB_TENANT", "evalhub")  # namespace where eval jobs run
EVALHUB_TOKEN = os.environ.get("EVALHUB_TOKEN") or _oc("whoami", "-t")

# --- The chatbot being evaluated ---
# This example talks to an OpenAI-compatible endpoint. Replace ask_chatbot() below
# with a call to your actual chatbot API.
CHATBOT_URL = os.environ.get("LITELLM_API_URL", "https://your-chatbot.example.com/v1")
CHATBOT_API_KEY = os.environ.get("LITELLM_API_KEY", "")
CHATBOT_MODEL = os.environ.get("LITELLM_MODEL", "llama-scout-17b")

# --- The judge model RAGAS uses to grade answers ---
# Can be any OpenAI-compatible endpoint; here we reuse the same gateway as the chatbot.
JUDGE_URL = os.environ.get("JUDGE_URL", CHATBOT_URL)
JUDGE_MODEL = os.environ.get("JUDGE_MODEL", CHATBOT_MODEL)
JUDGE_EMBEDDING_MODEL = os.environ.get("JUDGE_EMBEDDING_MODEL", "nomic-embed-text-v1-5")
JUDGE_EMBEDDING_URL = os.environ.get("JUDGE_EMBEDDING_URL", JUDGE_URL)
JUDGE_AUTH_SECRET = "judge-model-auth"  # k8s secret (key: api-key) in the tenant namespace

# --- S3 storage used to hand the dataset to the evaluation job ---
S3_ENDPOINT = os.environ.get("S3_ENDPOINT") or "https://" + _oc(
    "get", "route", "minio", "-n", "evalhub", "-o", "jsonpath={.spec.host}"
)
S3_ACCESS_KEY = os.environ.get("S3_ACCESS_KEY", "minio")
S3_SECRET_KEY = os.environ.get("S3_SECRET_KEY", "minio12345")
S3_BUCKET = os.environ.get("S3_BUCKET", "evals")
S3_SECRET_REF = "evalhub-test-data-s3"  # k8s secret the job uses to download the data

# --- MLflow experiment tracking ---
# All runs (stage 1 and stage 2) land in one experiment so you can compare them.
MLFLOW_URL = os.environ.get("MLFLOW_URL") or "https://" + _oc(
    "get", "route", "mlflow", "-n", "evalhub", "-o", "jsonpath={.spec.host}"
)
EXPERIMENT_NAME = os.environ.get("MLFLOW_EXPERIMENT", "rag-chatbot-eval")  # k8s secret the job uses to download the data

print(f"EvalHub: {EVALHUB_URL} (tenant: {EVALHUB_TENANT})")
print(f"Chatbot: {CHATBOT_MODEL} @ {CHATBOT_URL}")
print(f"Judge:   {JUDGE_MODEL} @ {JUDGE_URL}")


## 1. Curate the test set

Each entry pairs a question with a **reference answer** — what a domain expert considers
a good answer. This is the most valuable artifact you will build: lean on your team's
domain expertise, and grow it over time.

For a real evaluation aim for **30–50 questions** that represent real usage:

- the common cases your users actually ask,
- questions that require synthesizing across multiple documents,
- questions with nuanced answers,
- questions the chatbot has gotten wrong before.

The 10 questions below are illustrative placeholders (general agri-food domain) — replace
them with questions from your own domain.


In [ ]:
test_set = [
    {
        "question": "What does the DOP label mean on Italian food products?",
        "reference": "DOP (Denominazione di Origine Protetta, Protected Designation of Origin) certifies that a product is produced, processed, and prepared entirely in a specific geographic area, using recognized local know-how and following a strict production protocol.",
    },
    {
        "question": "What is the difference between DOP and IGP certifications?",
        "reference": "DOP requires that all production phases happen in the designated area, while IGP (Indicazione Geografica Protetta) only requires that at least one production phase takes place in the area.",
    },
    {
        "question": "What is crop rotation and why is it used?",
        "reference": "Crop rotation is the practice of growing different crops in sequence on the same land across seasons. It improves soil fertility, reduces pest and disease build-up, and limits the need for chemical inputs.",
    },
    {
        "question": "Which region of Italy produces Parmigiano Reggiano?",
        "reference": "Parmigiano Reggiano is produced in a delimited area covering the provinces of Parma, Reggio Emilia, Modena, and parts of Bologna and Mantua.",
    },
    {
        "question": "What is integrated pest management?",
        "reference": "Integrated pest management (IPM) is an approach that combines biological, cultural, physical, and chemical tools to control pests with minimal environmental impact, using pesticides only when monitoring shows they are needed.",
    },
    {
        "question": "What conditions must a product meet to be labelled organic in the EU?",
        "reference": "EU organic products must be grown without synthetic pesticides and fertilizers, without GMOs, respecting animal welfare standards, and be certified by an approved control body under EU Regulation 2018/848.",
    },
    {
        "question": "What is the purpose of the EU Common Agricultural Policy (CAP)?",
        "reference": "The CAP supports European farmers through income support, market measures, and rural development programmes, aiming to ensure food security, fair income for farmers, and sustainable management of natural resources.",
    },
    {
        "question": "How does drip irrigation save water compared to sprinkler systems?",
        "reference": "Drip irrigation delivers water directly to the plant root zone through emitters, reducing evaporation and runoff losses, and can cut water use significantly compared to sprinklers that wet the whole field surface.",
    },
    {
        "question": "What is agritourism?",
        "reference": "Agritourism is a form of tourism where visitors stay on working farms, combining hospitality with farm activities such as food tastings, harvest participation, and educational experiences, providing farmers an additional income source.",
    },
    {
        "question": "Why are bees important for agriculture?",
        "reference": "Bees pollinate a large share of food crops, including fruits, vegetables, and seed crops. Without pollinators, yields and quality of many crops would drop sharply, threatening food production and biodiversity.",
    },
]

print(f"{len(test_set)} questions in the test set")


## 2. Ask the chatbot

Run every question through the chatbot and record its responses. This is the **only**
integration point with your system: replace `ask_chatbot()` with a call to your chatbot's
API. The chatbot stays a black box — we don't care *how* it produces the answer yet.


In [ ]:
from openai import OpenAI

chatbot = OpenAI(base_url=CHATBOT_URL, api_key=CHATBOT_API_KEY)


def ask_chatbot(question: str) -> str:
    """Ask the chatbot one question. Replace this with your chatbot's API call."""
    completion = chatbot.chat.completions.create(
        model=CHATBOT_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer concisely in 2-3 sentences."},
            {"role": "user", "content": question},
        ],
        temperature=0,
    )
    return completion.choices[0].message.content.strip()


records = []
for i, item in enumerate(test_set, 1):
    answer = ask_chatbot(item["question"])
    records.append(
        {
            # RAGAS column names: user_input / response / reference
            "user_input": item["question"],
            "response": answer,
            "reference": item["reference"],
        }
    )
    print(f"[{i}/{len(test_set)}] {item['question'][:60]}")


In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 100)
df = pd.DataFrame(records)
df


## 3. Upload the dataset

EvalHub jobs read custom test data from S3-compatible storage: we upload the dataset as
JSONL, and the job downloads it (using the `evalhub-test-data-s3` secret) before the
evaluation starts.


In [ ]:
import json
from datetime import datetime, timezone

import boto3
from botocore.config import Config

run_tag = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
dataset_key = f"stage1/{run_tag}/dataset.jsonl"

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=S3_ACCESS_KEY,
    aws_secret_access_key=S3_SECRET_KEY,
    region_name="us-east-1",
    config=Config(request_checksum_calculation="when_required")
)

if S3_BUCKET not in [b["Name"] for b in s3.list_buckets()["Buckets"]]:
    s3.create_bucket(Bucket=S3_BUCKET)

jsonl = "\n".join(json.dumps(r) for r in records)
s3.put_object(Bucket=S3_BUCKET, Key=dataset_key, Body=jsonl.encode())
print(f"Uploaded {len(records)} rows to s3://{S3_BUCKET}/{dataset_key}")


## 4. Submit the evaluation job

The job tells EvalHub: run the `ragas` provider, on this dataset, with these metrics,
using this judge model. Note that `model` here is the **judge** — the LLM RAGAS uses to
grade answers — not the chatbot (whose answers are already in the dataset).


In [ ]:
from evalhub import ModelConfig, SyncEvalHubClient
from evalhub.models.api import (
    BenchmarkConfig,
    ExperimentConfig,
    JobSubmissionRequest,
    ModelAuth,
    S3TestDataRef,
    TestDataRef,
)

client = SyncEvalHubClient(
    base_url=EVALHUB_URL,
    auth_token=EVALHUB_TOKEN,
    tenant=EVALHUB_TENANT,
    timeout=60.0,
)

request = JobSubmissionRequest(
    name=f"rag-stage1-{run_tag}",
    description="Stage 1: answer correctness vs. reference answers",
    experiment=ExperimentConfig(name=EXPERIMENT_NAME),
    model=ModelConfig(
        url=JUDGE_URL,
        name=JUDGE_MODEL,
        auth=ModelAuth(secret_ref=JUDGE_AUTH_SECRET),
    ),
    benchmarks=[
        BenchmarkConfig(
            id="ragas_rag_default",
            provider_id="ragas",
            parameters={
                "metrics": ["factual_correctness", "answer_similarity", "nv_accuracy"],
                "embedding_model": JUDGE_EMBEDDING_MODEL,
                "embedding_url": JUDGE_EMBEDDING_URL
            },
            test_data_ref=TestDataRef(
                s3=S3TestDataRef(bucket=S3_BUCKET, key=dataset_key, secret_ref=S3_SECRET_REF)
            ),
        )
    ],
)

job = client.jobs.submit(request)
print(f"Job submitted: {job.id} (state: {job.state})")


## 5. Wait for results

EvalHub runs the evaluation as a Kubernetes job in the tenant namespace. Grading time
scales with dataset size and judge model speed — expect a couple of minutes for 10
questions, longer for a full 50-question set.


In [ ]:
final = client.jobs.wait_for_completion(job.id, timeout=1800, poll_interval=10)
print(f"Job {final.id} finished: {final.effective_state}")


In [ ]:
scores = final.results.benchmarks[0].metrics

summary = pd.DataFrame(
    [
        {"metric": name, "score": round(float(value), 3)}
        for name, value in sorted(scores.items())
    ]
)
summary


## 6. Reading the results

All metrics are averages over the test set, on a 0–1 scale (higher is better).

- **High `factual_correctness` + high `answer_similarity`** — the chatbot answers match
  the references both in facts and meaning. Retrieval tuning may not be urgent.
- **High `answer_similarity` but low `factual_correctness`** — the answers *sound* right
  but get facts wrong. Classic hallucination signal: time for Stage 2 to find out whether
  retrieval or generation is to blame.
- **Low across the board** — the chatbot is missing the expected answers entirely.
  Check whether the relevant documents are even in its knowledge base.

**Read the failures.** Aggregate scores tell you *whether* there is a problem; the
lowest-scoring questions tell you *what kind*. Re-run the worst questions through the
chatbot and look at the answers: are they hard retrieval problems? Ambiguous questions?
Model limitations? That tells you whether Stage 2 is worth setting up now.

**Make it a habit.** Keep the test set in version control, re-run this notebook before
and after every significant change (new documents, prompt changes, model upgrades), and
track the scores over time. A score that drops is a conversation worth having before the
change ships.

➡️ Continue with [`stage2_retrieval_metrics.ipynb`](stage2_retrieval_metrics.ipynb) to
add retrieval-aware metrics and find out *where* failures come from.


## Track your runs in MLflow

Because the job carried an `experiment` config, EvalHub logged this evaluation as an
MLflow run: the aggregate metrics, plus the **per-question scores** as `results.jsonl`
and `results.csv` artifacts (browse them in the MLflow UI — the URL is printed below).

This is what makes evaluation a habit instead of a one-off: every re-run of this
notebook appends a row here. Re-run before and after every significant change —
new documents, prompt tweaks, model upgrades — and a score that drops is a
conversation worth having before the change ships.


In [ ]:
import requests

MLFLOW_URL = os.environ.get("MLFLOW_URL") or "https://" + _oc(
    "get", "route", "mlflow", "-n", "", "-o", "jsonpath={.spec.host}"
)
MLFLOW_API = f"{MLFLOW_URL}/api/2.0/mlflow"

exp = requests.get(
    f"{MLFLOW_API}/experiments/get-by-name", params={"experiment_name": EXPERIMENT_NAME}
).json()["experiment"]
runs = requests.post(
    f"{MLFLOW_API}/runs/search",
    json={"experiment_ids": [exp["experiment_id"]], "max_results": 100},
).json().get("runs", [])

history = pd.DataFrame(
    [
        {
            "run": r["info"].get("run_name", r["info"]["run_id"][:8]),
            "when": pd.to_datetime(int(r["info"]["start_time"]), unit="ms").round("s"),
            **{m["key"]: round(m["value"], 3) for m in r["data"].get("metrics", [])},
        }
        for r in runs
    ]
).sort_values("when").reset_index(drop=True)

print(f"MLflow UI: {MLFLOW_URL}/#/experiments/{exp['experiment_id']}")
history
